In [1]:
import os
import pandas as pd
import boto3
from scripts.utils.db import engine
from sqlalchemy import text

from dataclasses import dataclass
from pydantic import BaseModel, Field

from langchain_aws import ChatBedrock
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.utilities import SQLDatabase


from sqlalchemy import create_engine
from langchain_aws import ChatBedrock
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool

/Users/konst/Documents/chatbot_ai_agent/chatbot_agent/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [8]:
import boto3

client = boto3.client("bedrock", region_name="us-east-1")

resp = client.list_inference_profiles()

for p in resp["inferenceProfileSummaries"]:
    print(p["inferenceProfileName"])
    print(p["inferenceProfileArn"])
    print("---")

US Anthropic Claude 3 Sonnet
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.anthropic.claude-3-sonnet-20240229-v1:0
---
US Anthropic Claude 3 Opus
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.anthropic.claude-3-opus-20240229-v1:0
---
US Anthropic Claude 3 Haiku
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.anthropic.claude-3-haiku-20240307-v1:0
---
US Meta Llama 3.2 11B Instruct
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.meta.llama3-2-11b-instruct-v1:0
---
US Meta Llama 3.2 3B Instruct
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.meta.llama3-2-3b-instruct-v1:0
---
US Meta Llama 3.2 90B Instruct
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.meta.llama3-2-90b-instruct-v1:0
---
US Meta Llama 3.2 1B Instruct
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.meta.llama3-2-1b-instruct-v1:0
---
US Anthropic Claude 3.5 Haiku
arn:aws:bedrock:us-east-1:355548799622:inference-profile/us.anthropic.cla

In [2]:
# --- Bedrock client ---
client = boto3.Session().client(
    "bedrock-runtime",
    region_name="us-east-1"
)
print(client.meta.endpoint_url)

https://bedrock-runtime.us-east-1.amazonaws.com


In [3]:
SYSTEM_PROMPT = """
You are a marketing data expert querying the 'marketing_data' table.

### DATABASE SCHEMA:
Table: marketing_data
Columns:
- year (BIGINT), quarter (TEXT), month (TEXT), week (BIGINT), date (TIMESTAMP)
- country (TEXT), media_category (TEXT), media_name (TEXT), communication (TEXT)
- campaign_category (TEXT), product (TEXT), campaign_name (TEXT)
- revenue (DOUBLE), cost (DOUBLE), profit (DOUBLE), roi (DOUBLE), margin (DOUBLE)
- quarter_number (1-4), month_number (1-12), month_name (TEXT)

Rules:
1. Use 'chat_history' to identify the specific metric (e.g., revenue, cost) from the previous turn.
2. If the user says "same but [condition]", ONLY change that specific condition in your SQL.

### TOOLS:
You can use tools to query the database. You must follow the ReAct format strictly.

### FORMAT INSTRUCTIONS:
To answer the user, you MUST use the following sequence:

Thought: Describe your plan to answer the question.
Action: [tool_name]
Action Input: [query or input for the tool]
Observation: [the result from the tool]

... (Repeat Thought/Action/Observation if needed)

Thought: I now know the final answer.
Final Answer: [The clear, concise summary of the data for the user]

### IMPORTANT RULES:
- If the user is vague (e.g., "revenue"), assume they want the SUM(revenue).
- When asked for a "same but [Year]", look at the previous SQL and only change the year filter.
- NEVER end a response without the prefix 'Final Answer:'.
"""

### LangChain

In [4]:
@dataclass
class MarketingResponse:
    """The final response to the user's marketing query."""
    answer: str  # The actual data (e.g., "Revenue was $10M")
    justification: str  # How you found it (e.g., "I summed the revenue column for 2023")
    metric_used: str  # e.g., "revenue"

In [5]:

db = SQLDatabase(engine)

llm = ChatBedrock(
    client=client,
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    model_kwargs={
        "temperature": 0,
        "system": SYSTEM_PROMPT
    }
)

agent_executor = create_sql_agent(
    llm=llm,
    db=db,
    verbose=False,           
    handle_parsing_errors=True,
    response_format=MarketingResponse,
    include_tables=['marketing_data'],
)

while True:
    
    q = input("You: ")
    if q.lower() in ["exit", "quit"]:
        break

    try:
        response = agent_executor.invoke({"input": q})
        
        print(f"Bot: {response['output']}")

    except Exception as e:
        print(f"An error occurred: {e}")

You:  Revenue for 2023


Bot: The total revenue for 2023 is $23,990,182.60.


You:  exit


In [97]:
tools = toolkit.get_tools()

for i, t in enumerate(tools, 1):
    print(f"{i}. {t.name}")
    print(f"   {t.description}\n")

1. sql_db_query
   Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

2. sql_db_schema
   Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

3. sql_db_list_tables
   Input is an empty string, output is a comma-separated list of tables in the database.

4. sql_db_query_checker
   Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



# LangGraph with Postgres as persistent checkpoint storage

In [6]:

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()


# --- LangGraph Nodes ---
def call_model(state: MessagesState):
    """The 'Brain': Decides to query SQL or answer the user."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + state["messages"]
    response = llm.bind_tools(tools).invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState):
    """Router: Check if the model called a SQL tool or is finished."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# --- Building the graph ---
workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent") # Loop back to analyze the SQL results

raw_uri = engine.url.render_as_string(hide_password=False)

pool_uri = raw_uri.replace("+psycopg", "")

# starting pool with the corrected string for psycopg 3
with ConnectionPool(conninfo=pool_uri, max_size=10, kwargs={"autocommit": True}) as pool:
    checkpointer = PostgresSaver(pool)
    checkpointer.setup()
    
    app = workflow.compile(checkpointer=checkpointer)

    config = {"configurable": {"thread_id": "marketing_session_001"}}
    
    print("--- Marketing Bot Active (Type 'exit' to quit) ---")
    while True:
        q = input("\nYou: ")
        if q.lower() in ["exit", "quit"]: break

        for event in app.stream({"messages": [{"role": "user", "content": q}]}, config):
            for value in event.values():
                if "messages" in value:
                    msg = value["messages"][-1]
                    if hasattr(msg, "content") and msg.content:
                        print(f"Bot: {msg.content}")

--- Marketing Bot Active (Type 'exit' to quit) ---



You:  Revenue for 2023


Bot: **Final Answer:** The total revenue for 2023 is $23,990,182.60.



You:  same butfor 2022


Bot: **Final Answer:** The total revenue for 2022 is $30,357,817.33.



You:  capital of france 


Bot: I'm a marketing data expert focused on querying the 'marketing_data' table. I can only help you with questions related to marketing metrics like revenue, cost, profit, ROI, campaigns, and other data from our marketing database.

I cannot answer general knowledge questions like the capital of France. 

Is there anything about the marketing data you'd like to explore? For example, I can help you analyze revenue, costs, campaign performance, or other marketing metrics by year, country, product, or campaign.



You:  exit
